### Experiments: BPE tokenizer for PAWS-Wiki

## 1. Training the tokenizer

Our classifier needs a way to segment text into tokens. We train a **Byte-Pair Encoding (BPE)** tokenizer on the PAWS training set.

BPE starts from single characters and repeatedly merges the most frequent pair of symbols until it reaches the target vocabulary size. Frequent words end up as a single token, while rare words (e.g. names) are split into sub-word pieces, so we almost never need an `[UNK]` token.

**Training data.** We use both sentences of every pair in the **train** split only (98,802 sentences), so no information from validation or test leaks into the tokenizer. PAWS is built from Wikipedia, so it is large enough for a vocabulary of this size.

In [5]:
from src.dataset import load_paws, tokenizer_corpus
from src.tokenization import train_tokenizer

dataset = load_paws()
corpus = tokenizer_corpus(dataset)   # train only, sentence1 + sentence2
print(f"{len(corpus):,} sentences in the tokenizer corpus")

98,802 sentences in the tokenizer corpus


## 2. Choosing the hyperparameters

Two hyperparameters:
- `vocab_size`: number of tokens in the vocabulary.
- `min_frequency`: a pair of symbols is only merged if it appears at least this many times.

We train several tokenizers and measure, on the **validation** sentences (unseen during training):
- `avg_tokens_per_sentence`: fewer tokens means the vocabulary groups more characters per token.
- `unk_rate`: fraction of tokens that are `[UNK]`. Close to 0 means the tokenizer covers new text.

Larger vocabularies give shorter sequences but sparser features, since many tokens appear only a few times.

In [ ]:
import pandas as pd
from src.tokenization import tokenizer_stats

val_texts = list(dataset["validation"]["sentence1"]) + list(dataset["validation"]["sentence2"])

# Train each tokenizer once and keep it, so the following cells can reuse it
tokenizers = {}
rows = []
for vocab_size in [1000, 4000, 8000]:
    for min_freq in [2, 5]:
        tokenizers[(vocab_size, min_freq)] = train_tokenizer(corpus, vocab_size, min_freq)
        rows.append({"min_frequency": min_freq, **tokenizer_stats(tokenizers[(vocab_size, min_freq)], val_texts)})
pd.DataFrame(rows)

**Observations.**
- `min_frequency` has **no effect** here (2 and 5 give identical rows): with these vocabulary sizes every merge already appears far more than 5 times, so the filter never applies and training stops when `vocab_size` is reached. We keep `min_frequency=2`, as in the tutorial.
- `vocab_size` matters: going from 1,000 (the tutorial's value) to 4,000 shortens sentences considerably (≈45 → ≈33 tokens), and 8,000 gives a smaller further gain (≈29).
- `[UNK]` is practically zero in all cases (<0.02%), so coverage is not a differentiating factor.

These statistics only describe the tokenizers; they do not tell us which one classifies best. That is decided later with the classifier's validation accuracy.

### 2.1. Inspecting how sentences are tokenized

The table above averages over the whole validation set. To see *how* text is actually split, we look at three validation sentences with each vocabulary size. `▁` marks the start of a word. We look for: are frequent words kept whole? Are rare words (names) split into meaningful pieces or into single characters?

In [ ]:
for i in [0, 5, 10]:
    sentence = dataset["validation"][i]["sentence1"]
    print(sentence)
    for vocab_size in [1000, 4000, 8000]:
        tokens = tokenizers[(vocab_size, 2)].encode(sentence).tokens
        print(f"  vocab={vocab_size:>5} ({len(tokens):>2} tokens): {' '.join(tokens)}")
    print()

**What to look for.** With a small vocabulary even common words are split (e.g. *husband*), which loses information; with a large one almost every word is a token, but many tokens are rare, so each has few training examples. Rare proper names are split in all cases.

## 3. Final tokenizer

In [ ]:
# Final tokenizer: reuse the one already trained above (no need to train it again)
final_vocab_size = 8000   # change if the classifier comparison favours another size
tokenizer = tokenizers[(final_vocab_size, 2)]
tokenizer.save(f"tokenizer_bpe_{final_vocab_size}.json")
print("vocabulary size:", tokenizer.get_vocab_size())

### 3.1. Example

A sentence from the validation set, split into BPE tokens. `▁` marks the start of a word. Frequent words stay whole, rare names are split into pieces.

In [4]:
example = dataset["validation"][0]["sentence1"]
print(example)
print(tokenizer.encode(example).tokens)

Bradd Crellin represented BARLA Cumbria on a tour of Australia with 6 other players representing Britain , also on a tour of Australia .
['▁Brad', 'd', '▁Cre', 'l', 'lin', '▁represented', '▁B', 'A', 'R', 'LA', '▁C', 'umb', 'ria', '▁on', '▁a', '▁tour', '▁of', '▁Australia', '▁with', '▁6', '▁other', '▁players', '▁represent', 'ing', '▁Britain', '▁,', '▁also', '▁on', '▁a', '▁tour', '▁of', '▁Australia', '▁.']


## 4. Representation spaces

An encoder maps one input to a vector. For a pair we encode each sentence with a base encoder and concatenate the two vectors (`PairEncoder`):

| Name | Base encoder (per sentence) |
|---|---|
| `TokenIndicator` | 1 if the token appears in the sentence, else 0 |
| `BagOfWords` | number of times each token appears in the sentence |

In [ ]:
from collections import OrderedDict
from src.encoders import TokenIndicator, BagOfWords, PairEncoder

encoders = {
    "TokenIndicator": PairEncoder(TokenIndicator(tokenizer)),
    "BagOfWords":     PairEncoder(BagOfWords(tokenizer)),
}

## 5. Cross-validation setup

Each PAWS example is a **pair** of sentences, so an input `x` is the tuple `(sentence1, sentence2)`. The code from the tutorial works with `(input, label)` pairs, so we build them in that format.

We run 5-fold cross-validation on the **train** split: the data is shuffled and split into 5 parts, and each model is trained 5 times, each time evaluating on a different part. The validation and test splits of PAWS are left untouched. The seed makes the shuffling reproducible, and all models use exactly the same folds so that the comparison is fair.

In [ ]:
from src import experiment, clslib, analysis
from src.dataset import paws_pairs

outcome_space = clslib.NamedClasses(["different meaning", "paraphrase"])
results_folder = "results"

my_seeds = [0]
num_folds = 5
my_cross_validation_sets = [
    experiment.cross_validation_split(paws_pairs(dataset["train"]), k=num_folds, seed=seed)
    for seed in my_seeds
]

## 6. Logistic regression (log-linear model)

We train one logistic regression per representation space and per fold, and record the accuracy on the held-out fold.

In [ ]:
my_classifiers = OrderedDict()

for enc_name, enc in encoders.items():
    name = f"logistic_{enc_name}"
    my_classifiers[name] = [
        experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=enc,
            model_option="logistic",
            save_to=f"{results_folder}/{name}/seed-{seed}",
            loglin_args={"max_iter": 100},
        )
        for run, seed in enumerate(my_seeds)
    ]

analysis.plot_box_and_points(analysis.make_data_frame(my_classifiers))

## 7. MLP (one hidden layer)

Same experiment, replacing the logistic regression with a neural network with one hidden layer (128 units, ReLU, dropout 0.1).

In [ ]:
mlp_args = {"hidden_dim": 128, "dropout": 0.1, "batch_size": 64,
            "learning_rate": 1e-4, "weight_decay": 1e-4, "num_epochs": 10}

for enc_name, enc in encoders.items():
    name = f"mlp_{enc_name}"
    my_classifiers[name] = [
        experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=enc,
            model_option="mlp",
            save_to=f"{results_folder}/{name}/seed-{seed}",
            smlp_args=mlp_args,
        )
        for run, seed in enumerate(my_seeds)
    ]

analysis.plot_box_and_points(analysis.make_data_frame(my_classifiers))